# S6_01 — MCP 서버 기초: `DocumentMCP`

**Skilljar Section 6** — Building with the Claude API: Model Context Protocol  
**다루는 레슨**: L01 (MCP 소개) · L03 (프로젝트 셋업) · L04 (도구 정의)

## 강의노트 매핑 (`Week_07.md`)

본 노트북은 강의노트의 다음 절들을 코드로 재현한다.

| 절 | 주제 | 라인 |
|---|---|---|
| §1.1 | MCP 란 무엇인가 — 개발 환경의 USB-C | 206-310 |
| §1.3 | 프로젝트 셋업: CLI 챗봇과 MCP 서버 | 437-533 |
| §1.4 | `@mcp.tool()` 데코레이터로 도구를 정의하는 법 | 535-666 |

## 사전 준비
다음 패키지가 설치되어 있어야 한다.
- Python 3.11 이상.
- `pip install "mcp[cli]" pydantic` 으로 SDK 와 검증 라이브러리 설치.
- (권장) 서버 실행 도구로 `uv` 를 설치한다: `pip install uv`.

## 학습 목표
본 노트북을 마치면 다음을 할 수 있어야 한다.
1. 이름이 `"DocumentMCP"` 이고 로그 레벨이 `"ERROR"` 인 서버를 단 한 줄로 초기화한다.
2. Skilljar 예제와 동일한 6개 문서를 메모리 딕셔너리로 구성한다.
3. 두 함수에 데코레이터를 적용해 도구 `read_doc_contents` 와 `edit_document` 를 등록한다.
4. 자동 생성된 JSON 스키마를 직접 확인하고, 도구를 인-프로세스로 호출해 동작을 검증한다.


## §1. MCP 란 무엇인가 — 표준화된 도구 사용

**MCP (Model Context Protocol)** 는 LLM 애플리케이션과 외부 도구·데이터 소스를 연결하는 **개방형 프로토콜** 이다. Skilljar 의 첫 번째 레슨은 MCP 를 다음과 같이 한 문장으로 요약한다.

> 도구의 정의와 실행이라는 부담을, 우리 서버에서 전용 MCP 서버 쪽으로 옮기는 방법.

곧 핵심은 **"도구를 누가 만들고 누가 유지보수할 것인가"** 라는 책임의 이관이다. 이전 4주차에서 우리는 도구의 스키마와 함수를 모두 직접 작성했었다. MCP 는 그 작업을 **다른 누군가가 미리 패키징해 놓은 서버** 에 위임하게 해 준다.

### 4주차의 도구 사용 (Tool Use) 과 7주차의 MCP 비교

표면적으로 두 방식은 비슷해 보이지만, **도구가 어디에 살고, 누가 스키마를 작성하며, 다른 앱에서 재사용 가능한가** 라는 질문에 대한 답이 다르다.

| 관점 | 4주차의 도구 사용 | 7주차의 MCP |
|---|---|---|
| 도구의 위치 | 우리 앱 안에 직접 둔다 | 별도 MCP 서버 안에 둔다 |
| 스키마 작성자 | 본인이 손으로 JSON 을 작성 | 서버 작성자가 데코레이터로 자동 생성 |
| 재사용성 | 한 앱 안에서만 쓸 수 있다 | MCP 호환 클라이언트 어디서나 쓸 수 있다 |
| 통신 방식 | 같은 프로세스 안 함수 호출 | stdio · HTTP · WebSocket 등 다양한 전송 |
| 대표 예시 | 4주차 날씨 도구 | 7주차의 `DocumentMCP` 서버 (`read_doc_contents`, `edit_document`) |

이 표가 보여주듯, MCP 는 단순히 "도구를 부르는 새로운 방법" 이 아니라 **도구의 소유 구조와 배포 모델 전체를 바꾸는** 변화다. 강의노트 §1.1 (라인 206-310) 에서 자세히 설명한 *"도구 함수 문제"* — 즉 거대한 외부 서비스를 통합하기 위해 수십 개의 스키마와 함수를 일일이 손으로 짜야 하던 문제 — 가 바로 MCP 가 해결하려는 문제다.


## §2. 셋업 — 본 노트북에서 사용할 라이브러리

MCP 서버를 만드는 데는 단 세 줄의 import 가 필요하다. 첫째는 매개변수 설명을 붙이기 위한 Pydantic 의 `Field` 클래스, 둘째는 서버 본체인 `FastMCP` 클래스, 셋째는 이후 S6_05 에서 사용할 프롬프트 메시지 클래스다. 셋째 줄은 본 노트북에서는 직접 쓰지 않지만, 같은 파일을 그대로 다음 노트북들이 공유하기 때문에 일관성을 위해 같이 가져온다.


In [ ]:
# Week_07.md §1.4 라인 ~549 — FastMCP import 는 단 한 줄
from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base  # S6_05 에서 사용; 일관성을 위해 여기서도 import


## §3. FastMCP 서버 초기화

MCP 의 Python SDK 가 가장 인상적인 부분 중 하나는 **서버 한 대를 단 한 줄** 로 초기화한다는 점이다. 첫 번째 인자인 `"DocumentMCP"` 는 서버의 이름이며 클라이언트 로그와 Inspector UI 에서 식별자로 표시된다. 두 번째 인자인 `log_level="ERROR"` 는 표준 출력을 깨끗하게 유지하기 위해 사용한다.

왜 표준 출력을 깨끗하게 유지해야 할까. stdio 전송 방식은 서버와 클라이언트가 **표준 입력과 표준 출력을 그대로 JSON-RPC 메시지의 통로** 로 사용한다. 따라서 서버 코드 어딘가에서 무심코 `print()` 한 줄이 끼어들면 그 출력이 프로토콜 스트림에 섞여 클라이언트가 메시지를 파싱하지 못한다. 로그 레벨을 에러로 제한하면 일반 정보 로그가 표준 출력을 어지럽히는 사고를 막을 수 있다.


In [ ]:
# Week_07.md §1.4 라인 ~551 — 서버 초기화
mcp = FastMCP("DocumentMCP", log_level="ERROR")

print(f"Server name: {mcp.name}")


## §4. 메모리 안에 보관하는 문서 컬렉션

Skilljar 프로젝트는 예제를 단순하게 유지하기 위해 데이터베이스 없이 **6개의 고정 문서** 를 Python 딕셔너리에 직접 담아 사용한다. 키는 문서 식별자, 값은 문서의 본문 텍스트다.

주의 깊게 보아야 할 부분은 6개 문서의 **분위기** 다. *Angela Smith, P.E.* (전문 엔지니어) 의 진술서, *20m condenser tower* 의 상태 보고서, 프로젝트의 예산과 지출, 시스템의 미래 성능 전망, 프로젝트 실행 계획, 장비의 기술 사양 — 모두 **건축·엔지니어링 도메인** 의 어휘로 구성되어 있다. 이는 우연이 아니다. 강의노트 §1.4 가 강조하듯, Skilljar 는 엔지니어링 학생들이 도메인을 친숙하게 느끼도록 의도적으로 이런 어휘를 골랐다. 본 강의의 §2.7 에서는 이 패턴을 그대로 가져와 **KDS 조문** 이나 **Midas 해석 결과** 를 담는 문서 저장소로 확장한다.


In [ ]:
# cli_project/mcp_server.py:8-15 — 6개 문서 시드 (원본 그대로)
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditures.",
    "outlook.pdf": "This document presents the projected future performance of the system.",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment.",
}

print(f"Loaded {len(docs)} documents")
for doc_id in docs:
    print(f"  - {doc_id}")


## §5. 첫 번째 도구 — `read_doc_contents`

이제 본 노트북에서 가장 중요한 부분으로 들어간다. `@mcp.tool` 데코레이터는 표면적으로는 그저 함수에 붙이는 한 줄짜리 표시지만, 실제로는 **두 가지 큰 일** 을 동시에 한다.

첫째, 그 데코레이터는 일반적인 Python 함수를 **MCP 런타임이 디스패치할 수 있는 도구로 등록** 한다. 즉 클라이언트가 `tools/list` 요청을 보내면 이 함수의 이름과 설명이 응답에 포함된다.

둘째 — 그리고 더 중요한 것이지만 — 데코레이터는 **함수의 타입 힌트와 Pydantic Field 의 description 을 읽어 JSON 스키마를 자동으로 생성** 한다. 이는 4주차에서 도구를 사용할 때 우리가 손으로 작성하던 그 긴 JSON 객체를 자동으로 만들어 준다는 뜻이다. Claude 가 어떤 도구를 부를지 결정할 때 보는 것이 바로 이 자동 생성된 스키마다.

여기서 절대 가벼이 보면 안 되는 사실: 데코레이터의 `description` 인자와 각 매개변수에 붙은 `Field(description=...)` 의 설명문은 **단순한 주석이 아니다**. 이 문장들이 곧 Claude 가 도구를 고르는 근거가 된다. 모호한 설명은 모호한 호출을, 명료한 설명은 정확한 호출을 만든다. 이 차이는 §9.5 에서 직접 실험으로 확인한다.

> [!tip] 강의노트 §1.4 의 핵심 문장 (라인 660 부근)
> 타입 힌트만으로 스키마가 만들어진다. `doc_id: str = Field(description="...")` 한 줄이, 4주차에서 손으로 쓰던 JSON 스키마 전체 — 즉 객체 타입 선언, 속성 목록, 필수 필드 표시 — 를 모두 대체한다.


In [ ]:
# cli_project/mcp_server.py:17-28 — read_doc_contents 도구 (원본 그대로)
# Week_07.md §1.4 라인 ~586-598
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string",
)
def read_document(
    doc_id: str = Field(description="ID of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


print("read_doc_contents registered.")


## §6. 두 번째 도구 — `edit_document`

두 번째 도구는 매개변수가 셋이라서 첫 도구보다 조금 더 흥미롭다. `doc_id` 는 어떤 문서를 편집할지 가리키고, `old_str` 는 찾을 텍스트, `new_str` 는 그 자리에 넣을 새 텍스트다. 매개변수마다 별도의 `Field(description=...)` 가 붙어 있다는 점이 핵심이다.

Claude 는 이 자동 생성된 스키마를 보고 두 가지 판단을 내린다. 첫째, `old_str` 의 설명에 *공백을 포함해 정확히 일치해야 한다* 라고 적혀 있으므로 도구 호출 시 공백을 함부로 잘라내지 않는다. 둘째, `old_str` 와 `new_str` 라는 매개변수 이름과 각 설명이 서로 다른 역할 — 검색 대상과 대체 텍스트 — 을 분명히 구분해 주므로 두 인자를 헷갈려 바꾸어 넣지 않는다.

구현 자체는 Python 의 내장 메서드 `str.replace()` 한 줄에 불과하다. 이 단순함이 바로 MCP 의 매력이다 — 도구 본체는 평범한 Python 함수이고, 그 함수를 "외부에서 부를 수 있는 표준화된 도구" 로 만드는 일은 모두 데코레이터가 처리한다. 더 큰 문서 저장소나 복잡한 편집 로직으로 확장하더라도 같은 패턴이 그대로 통한다.


In [ ]:
# cli_project/mcp_server.py:31-44 — edit_document 도구 (원본 그대로)
# Week_07.md §1.4 라인 ~607-621
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace"),
    new_str: str = Field(description="The text to insert in place of the old text"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)


print("edit_document registered.")


## §7. 자동 생성된 스키마 직접 확인 — `mcp.list_tools()`

이제 정말로 데코레이터가 우리 대신 JSON 스키마를 만들어 주었는지 확인할 차례다. 아래 셀은 FastMCP 런타임에 묻는다 — 어떤 도구가 등록되어 있고, Pydantic 이 우리의 타입 힌트를 어떤 스키마로 변환했는가. 우리는 JSON 한 줄도 직접 쓰지 않았지만, 결과는 OpenAI 와 Claude 의 `tools` 인자가 그대로 받아들이는 표준 형식이다.

Jupyter 는 셀 안에서 최상위 `await` 를 그대로 사용할 수 있게 해 준다. 따라서 일부러 이벤트 루프를 만들거나 `asyncio.run()` 을 호출할 필요 없이, async 함수를 곧바로 await 할 수 있다.


In [ ]:
# Week_07.md §1.4 라인 ~662 — 자동 생성된 스키마 검증
import json as _json

tools = await mcp.list_tools()
for t in tools:
    print(f"--- {t.name} ---")
    print(f"description: {t.description}")
    print(f"schema: {_json.dumps(t.inputSchema, indent=2)}")
    print()


## §8. 도구를 직접 호출하기 — `mcp.call_tool`

FastMCP 는 stdio 전송 계층을 거치지 않고 같은 프로세스 안에서 도구를 곧바로 호출하는 **인-프로세스 `call_tool`** 메서드를 제공한다. 이는 디버깅과 단위 테스트에 매우 유용하다 — 서브프로세스를 띄우고 JSON-RPC 핸드셰이크를 거치지 않아도 도구의 동작만 따로 떼어 검증할 수 있기 때문이다.

S6_03 노트북에서 만들 `MCPClient.call_tool` 은 같은 호출을 **JSON-RPC 메시지로 변환해 stdio 를 거쳐** 서버에 전달한다. 즉 여기서의 직접 호출과 S6_03 의 클라이언트 호출은 **같은 의미를 가지지만 통신 경로만 다른** 짝꿍이다. 둘이 같은 결과를 내는지 비교해 보면 MCP 가 추상화하는 "왕복" 의 정체가 분명해진다.


In [ ]:
# 인-프로세스 직접 호출 — MCP 클라이언트 호출과 동일한 형태
result = await mcp.call_tool("read_doc_contents", {"doc_id": "plan.md"})
print("read_doc_contents('plan.md') ->")
print(result)


## §9. 검증 연쇄 — 편집한 뒤 다시 읽기

다음 두 노트북에서도 똑같이 반복할 표준 시나리오다. 단 하나의 문서를 골라 그 내용을 바꾼 뒤, 정말로 바뀌었는지 다시 읽어 확인하는 세 단계 흐름이다.

1. `read_doc_contents("plan.md")` 로 원본의 본문을 출력한다.
2. `edit_document("plan.md", "outlines", "describes")` 로 단어 하나를 바꾼다.
3. 다시 `read_doc_contents("plan.md")` 로 변경 결과를 출력한다.

이 세 단계는 S6_02 의 Inspector UI 에서도, S6_03 의 클라이언트 코드에서도 그대로 반복된다. **같은 작업을 세 가지 다른 인터페이스로 본다** 는 점이 중요하다 — 인-프로세스 직접 호출, 브라우저 UI, 그리고 stdio 위의 JSON-RPC. 어느 경로를 거쳐도 결과는 같아야 한다.

마지막에는 다음 노트북들이 공유하는 상태가 깨지지 않도록 변경을 **원상 복구** 한다.


In [ ]:
# 1단계 — 원본
before = await mcp.call_tool("read_doc_contents", {"doc_id": "plan.md"})
print("BEFORE:", before)

# 2단계 — 편집
await mcp.call_tool(
    "edit_document",
    {"doc_id": "plan.md", "old_str": "outlines", "new_str": "describes"},
)

# 3단계 — 다시 읽기
after = await mcp.call_tool("read_doc_contents", {"doc_id": "plan.md"})
print("AFTER: ", after)

# 다음 노트북을 위해 원상 복구
await mcp.call_tool(
    "edit_document",
    {"doc_id": "plan.md", "old_str": "describes", "new_str": "outlines"},
)


## §9.5. Field description 실험 — Claude 의 도구 선택에 미치는 영향

강의노트 `Week_07.md` 라인 662-664 의 약속을 본 절에서 직접 실험한다. 그 약속은 다음과 같다.

> Pydantic Field 의 description 이 Claude 의 도구 선택에 어떻게 영향을 주는지도 실험한다.

데코레이터 인자의 `description` 과 `Field(description=...)` 는 **장식이 아니다**. 이 문장들은 Claude 가 어떤 도구를 호출할지 결정할 때 참조하는 **유일한 자연어 단서** 다. 이름이 같고 시그니처가 같아도, description 이 모호하면 Claude 는 잘못된 도구를 부르거나 호출 자체를 포기할 수 있다.

**실험 설계**: 함수 본체와 시그니처는 사실상 같지만 description 만 극단적으로 다른 두 도구를 만든다. 한쪽은 `"get something"` 처럼 거의 정보를 주지 않고, 다른 한쪽은 Skilljar 의 표준 패턴인 `"Read the contents of a document and return it as a string"` 처럼 정확한 행동을 명시한다. 그런 다음 두 서버에 각각 `list_tools()` 를 호출해 자동 생성된 스키마의 description 필드가 어떻게 다른지 비교한다.

**왜 중요한가**: 자동 생성된 스키마의 description 은 그대로 Claude 의 도구 선택 프롬프트에 들어간다. 모호한 단어들이 들어가면 Claude 의 추론이 흐려지고, 명료한 동사·명사가 들어가면 Claude 는 정확히 의도한 도구를 부른다. 이는 도구 호출 정확도를 높이는 가장 값싸고 가장 확실한 방법 중 하나다.

> [!finding] 핵심 통찰
> 모호한 description 인 `"get something"` 과 명시적 description 인 `"Read the contents of a document and return it as a string"` 은 같은 함수 본체를 가리키지만, Claude 의 입장에서는 **완전히 다른 도구** 처럼 읽힌다. 이 차이가 실제 호출 정확도의 차이를 만든다.


In [ ]:
# Week_07.md §1.4 라인 ~660 [!tip] — description 이 Claude 의 도구 선택에 미치는 영향 실험
# 강의노트 [!action] 약속: "Pydantic Field 의 description 이 Claude 의 도구 선택에 어떻게
# 영향을 주는지 실험한다."

# 비교 1: 모호한 description
mcp_vague = FastMCP("VagueDemo", log_level="ERROR")

@mcp_vague.tool(name="get_thing", description="get something")
def vague_get(item: str = Field(description="thing")):
    return f"got {item}"

# 비교 2: 명시적 description (Skilljar 패턴)
mcp_clear = FastMCP("ClearDemo", log_level="ERROR")

@mcp_clear.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string",
)
def clear_get(doc_id: str = Field(description="ID of the document to read")):
    return f"contents of {doc_id}"

# 자동 생성된 스키마 비교
import json as _json2

print("=== 모호한 버전 ===")
for t in await mcp_vague.list_tools():
    print(_json2.dumps(
        {"name": t.name, "description": t.description, "schema": t.inputSchema},
        indent=2, ensure_ascii=False,
    ))

print("\n=== 명시적 버전 (Skilljar) ===")
for t in await mcp_clear.list_tools():
    print(_json2.dumps(
        {"name": t.name, "description": t.description, "schema": t.inputSchema},
        indent=2, ensure_ascii=False,
    ))

# 결론: 이름·시그니처·반환값이 동일해도, description 의 명료도가 Claude 의
# 도구 선택 정확도를 결정한다. 모호하면 Claude 가 다른 도구를 부르거나 호출
# 자체를 포기할 수 있다.


## §10. 다음 노트북을 위해 `mcp_server.py` 저장하기

다음 두 노트북 — S6_02 의 Inspector 와 S6_03 의 `MCPClient` — 는 같은 폴더에 **실행 가능한 서버 파일** 이 있다는 것을 전제로 한다. 본 셀은 우리가 지금까지 본 노트북에서 만든 그 서버를 그대로 직렬화해 `mcp_server.py` 라는 파일로 저장한다.

저장하는 내용에는 본 노트북에서 다룬 두 도구뿐 아니라, 다음에 다룰 **리소스(`docs://documents`)** 와 **프롬프트(`format`)** 정의도 함께 포함된다. 이렇게 한 번에 통합 파일을 저장하면 S6_04 와 S6_05 에서 새 코드 없이 곧바로 그 기능들을 호출할 수 있다.

저장된 파일은 Skilljar 의 `cli_project/mcp_server.py` 와 동일하므로, 명령줄에서 `mcp dev mcp_server.py` 로 인스펙터를 띄우거나 `python mcp_server.py` 로 직접 실행해도 같은 동작을 한다.


In [ ]:
import os

SERVER_CODE = '''from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("DocumentMCP", log_level="ERROR")


docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project\\'s budget and expenditures.",
    "outlook.pdf": "This document presents the projected future performance of the system.",
    "plan.md": "The plan outlines the steps for the project\\'s implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment.",
}


@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string",
)
def read_document(
    doc_id: str = Field(description="ID of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id{doc_id} not found")
    return docs[doc_id]


@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="Teh text to replace. Must match exactly, including white space"),
    new_str: str = Field(description="The text to insert in place of the old text"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)


@mcp.resource(
    "docs://documents",
    mime_type="application/json",
)
def list_docs() -> list[str]:
    return list(docs.keys())


@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain",
)
def fetch(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


@mcp.prompt(
    name="format",
    description="Rewrite a document in markdown format",
)
def format_document(
    doc_id: str = Field(description="ID of the document to format")
) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat the following document in markdown format.

    The id o the document you need to format is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary.
    Feel free to add in examples to clarify the content.
    Use the \\'edit_document\\' tool to edit the document. After the documnet has been Do not leave anything out. The entire contents of the document should be included in the reformatted version.


    Use the \\'edit_document\\' tool to edit the document. After the documnet has been edited, return the full contents of the reformatted document as the final answer to this prompt.
    """

    return [base.UserMessage(prompt)]


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("mcp_server.py", "w", encoding="utf-8") as f:
    f.write(SERVER_CODE)

print(f"Wrote: {os.path.abspath('mcp_server.py')}")
print(f"Size: {os.path.getsize('mcp_server.py')} bytes")


## §11. 다음 단계 안내

본 노트북을 마쳤다면 다음 노트북들로 자연스럽게 이어진다.

- **S6_02 (Inspector)**: 명령 `mcp dev mcp_server.py` 로 브라우저 기반 인스펙터를 띄우고 도구를 UI 에서 직접 실행한다. 브라우저를 열 수 없는 환경에서는 stdio 위로 JSON-RPC 를 직접 보내는 대체 경로도 함께 다룬다.
- **S6_03 (Client)**: `async with` 구문으로 서브프로세스를 자동 정리하는 `MCPClient` 클래스를 만든다. 같은 서버에 실제 stdio 로 연결해 도구 목록을 가져오고 호출한다.
- **S6_04 / S6_05**: 본 노트북이 저장한 `mcp_server.py` 위에 리소스 (`docs://documents`) 와 프롬프트 (`format`) 호출을 추가한다.
- **`structural/` 트랙**: `DocumentMCP` 의 패턴을 그대로 가져와, 6개 문서 자리에 KDS 조문이나 Midas 해석 결과를 담는 도메인 응용 노트북으로 확장한다.
